In [1]:
import sqlite3
import pandas as pd
import numpy as np


In [2]:

conn = sqlite3.connect('tmdb_movies.db')
movies = pd.read_sql_query("SELECT * FROM movies", conn)
genres = pd.read_sql_query("SELECT * FROM genres", conn)
movie_genres = pd.read_sql_query("SELECT * FROM movie_genres", conn)
cast = pd.read_sql_query("SELECT * FROM cast", conn)
crew = pd.read_sql_query("SELECT * FROM crew", conn)
movie_keywords = pd.read_sql_query("SELECT * FROM movie_keywords", conn)
conn.close()


In [3]:

# Q1: Unique counts across all tables
print("Q1 Unique Counts:")
print({
    'Unique Movies': movies['movie_id'].nunique(),
    'Unique Genres': genres['genre_id'].nunique(),
    'Unique Cast Members': cast['person_id'].nunique(),
    'Unique Crew Members': crew['person_id'].nunique()
})


Q1 Unique Counts:
{'Unique Movies': 2503, 'Unique Genres': 19, 'Unique Cast Members': 11300, 'Unique Crew Members': 2848}


In [4]:

# Q2: Movies with missing or zero budget/revenue
zero_fin = movies[(movies['budget'] <= 0) | (movies['revenue'] <= 0)]
print(f"\nQ2: {len(zero_fin)} movies have 0/missing budget or revenue. Decision: Exclude these rows from ROI/Profit calculations so 0 placeholders don't skew financial metrics, while keeping them for rating/runtime/cast analyses.")



Q2: 133 movies have 0/missing budget or revenue. Decision: Exclude these rows from ROI/Profit calculations so 0 placeholders don't skew financial metrics, while keeping them for rating/runtime/cast analyses.


In [5]:

# Q3: Profit and ROI for valid budget & revenue movies
valid_fin = movies[(movies['budget'] > 0) & (movies['revenue'] > 0)].copy()
valid_fin['profit'] = valid_fin['revenue'] - valid_fin['budget']
valid_fin['roi_pct'] = ((valid_fin['profit'] / valid_fin['budget']) * 100).round(2)
print("\nQ3 Sample Profit & ROI:")
display(valid_fin[['title', 'budget', 'revenue', 'profit', 'roi_pct']].head(10))



Q3 Sample Profit & ROI:


,title,budget,revenue,profit,roi_pct
0,Interstellar,165000000,746606706,581606706,352.49
1,Inception,160000000,839030630,679030630,424.39
2,The Avengers,220000000,1518815515,1298815515,590.37
3,The Dark Knight,185000000,1004558444,819558444,443.00
4,Avatar,237000000,2923706026,2686706026,1133.63
5,Deadpool,58000000,782837347,724837347,1249.72
6,Fight Club,63000000,100853753,37853753,60.09
7,Avengers: Infinity War,300000000,2052415039,1752415039,584.14
8,The Shawshank Redemption,25000000,28341469,3341469,13.37
9,Pulp Fiction,8000000,213928762,205928762,2574.11


In [6]:

# Q4: Top 15 movies by ROI (budget >= $1M)
top_roi = valid_fin[valid_fin['budget'] >= 1_000_000].sort_values(by='roi_pct', ascending=False).head(15)
print("\nQ4 Top 15 Movies by ROI (Budget >= $1M):")
display(top_roi[['title', 'budget', 'revenue', 'profit', 'roi_pct']])



Q4 Top 15 Movies by ROI (Budget >= $1M):


,title,budget,revenue,profit,roi_pct
1767,Dragon Ball Super: Broly,1000000,125002821,124002821,12400.28
535,Snow White and the Seven Dwarfs,1488423,184925486,183437063,12324.26
1791,The Rocky Horror Picture Show,1400000,171181400,169781400,12127.24
467,Rocky,1000000,117253345,116253345,11625.33
1227,Gone with the Wind,4000000,402352579,398352579,9958.81
716,The Jungle Book,4000000,378000000,374000000,9350.00
635,Cinderella,2900000,263600000,260700000,8989.66
372,Saw,1200000,104045735,102845735,8570.48
719,One Hundred and One Dalmatians,3600000,303000000,299400000,8316.67
263,E.T. the Extra-Terrestrial,10500000,797307407,786807407,7493.40


In [7]:

# Q5: Movies released per year & peak year
movies['release_year'] = pd.to_datetime(movies['release_date'], errors='coerce').dt.year
yearly_counts = movies['release_year'].value_counts().sort_index()
print(f"\nQ5 Peak Release Year: {int(yearly_counts.idxmax())} with {yearly_counts.max()} movies.")



Q5 Peak Release Year: 2016 with 121 movies.


In [8]:

# Q6: Runtime statistical outliers (IQR method)
valid_runtime = movies[movies['runtime'] > 0]
Q1, Q3 = valid_runtime['runtime'].quantile([0.25, 0.75])
IQR = Q3 - Q1
outliers = valid_runtime[(valid_runtime['runtime'] < Q1 - 1.5 * IQR) | (valid_runtime['runtime'] > Q3 + 1.5 * IQR)]
print(f"\nQ6 Runtime Outliers ({len(outliers)} movies outside [{Q1 - 1.5*IQR:.1f}, {Q3 + 1.5*IQR:.1f}] mins):")
display(outliers[['title', 'runtime']].sort_values(by='runtime', ascending=False).head(10))



Q6 Runtime Outliers (60 movies outside [58.5, 166.5] mins):


,title,runtime
324,Zack Snyder's Justice League,242
1227,Gone with the Wind,238
829,Once Upon a Time in America,229
1610,Lawrence of Arabia,228
1774,Ben-Hur,212
587,The Irishman,209
1277,Seven Samurai,207
1304,Killers of the Flower Moon,206
187,The Godfather Part II,202
20,The Lord of the Rings: The Return of the King,201


In [9]:
# Q7: Primary genre & average popularity
primary_genre_map = movie_genres.drop_duplicates(subset=['movie_id'], keep='first').merge(genres, on='genre_id')
movies_pg = movies.merge(primary_genre_map[['movie_id', 'genre_name']], on='movie_id', how='left')
movies_pg['genre_name'] = movies_pg['genre_name'].fillna('Unknown')
print("\nQ7 Average Popularity by Primary Genre:")
display(movies_pg.groupby('genre_name')['popularity'].mean().sort_values(ascending=False).round(2))



Q7 Average Popularity by Primary Genre:


genre_name
Science Fiction    21.26
Music              18.32
Animation          16.81
Family             16.71
Action             16.39
War                15.47
Horror             14.85
Adventure          14.79
Fantasy            13.02
Mystery            12.43
Western            12.32
Thriller           11.29
Drama              11.28
Crime              10.74
History            10.41
Comedy              9.98
Romance             9.98
Unknown             9.37
Documentary         7.73
Name: popularity, dtype: float64

In [10]:

# Q8: Top 10 most prolific actors & their average movie rating
actor_stats = cast.merge(movies[['movie_id', 'vote_average']], on='movie_id') \
                  .groupby(['person_id', 'actor_name']) \
                  .agg(movie_count=('movie_id', 'nunique'), avg_rating=('vote_average', 'mean')) \
                  .sort_values(by='movie_count', ascending=False).head(10)
print("\nQ8 Top 10 Prolific Actors & Average Rating:")
display(actor_stats.round(2))



Q8 Top 10 Prolific Actors & Average Rating:


,,movie_count,avg_rating
person_id,actor_name,,
2231,Samuel L. Jackson,39,6.94
287,Brad Pitt,38,7.26
380,Robert De Niro,38,7.23
85,Johnny Depp,37,6.86
31,Tom Hanks,33,7.29
5293,Willem Dafoe,32,7.11
1245,Scarlett Johansson,32,7.06
13240,Mark Wahlberg,32,6.63
500,Tom Cruise,31,7.04


In [11]:

# Q9: Duplicate movie titles (remakes/reboots)
dup_titles = movies[movies.duplicated(subset=['title'], keep=False)].sort_values(by=['title', 'release_year'])
print(f"\nQ9 Duplicate Movie Titles ({len(dup_titles)} rows):")
display(dup_titles[['movie_id', 'title', 'release_year', 'vote_average']])



Q9 Duplicate Movie Titles (100 rows):


,movie_id,title,release_year,vote_average
878,377,A Nightmare on Elm Street,1984,7.350
1928,23437,A Nightmare on Elm Street,2010,5.525
261,812,Aladdin,1992,7.657
339,420817,Aladdin,2019,7.081
792,12092,Alice in Wonderland,1951,7.188
...,...,...,...,...
1737,60935,The Thing,2011,6.253
744,861,Total Recall,1990,7.324
860,64635,Total Recall,2012,6.033
2500,1368168,Transformers,1987,7.900


In [12]:

# Q10: Popularity vs Vote Count anomalies (Z-score divergence)
movies['pop_z'] = (movies['popularity'] - movies['popularity'].mean()) / movies['popularity'].std()
movies['vote_z'] = (movies['vote_count'] - movies['vote_count'].mean()) / movies['vote_count'].std()
movies['divergence'] = (movies['pop_z'] - movies['vote_z']).abs()
print("\nQ10 Popularity vs Vote Count Anomalies:")
display(movies.sort_values(by='divergence', ascending=False)[['title', 'release_year', 'popularity', 'vote_count', 'divergence']].head(10))



Q10 Popularity vs Vote Count Anomalies:


,title,release_year,popularity,vote_count,divergence
2388,Disclosure Day,2026,394.3346,2328,21.798752
46,Spider-Man: No Way Home,2021,438.4757,22429,20.182803
1315,Obsession,2026,364.8722,4203,19.793731
2248,Backrooms,2026,258.4743,2483,14.265255
713,Project Hail Mary,2026,162.7023,6750,8.116246
62,Spider-Man,2002,180.5936,20991,6.232464
1465,Michael,2026,108.7248,3846,5.721196
2452,Mortal Kombat II,2026,99.5231,2260,5.532897
1707,The Super Mario Galaxy Movie,2026,97.1210,3343,5.181862
35,Spider-Man: Homecoming,2017,169.0857,23456,5.099920
